In [ ]:
# Copyright (C) 2025 ETH Zurich, Moritz Thürlemann, and other AMP contributors

from rdkit import Chem
from rdkit.Chem import Draw
import json

methyl_capped_side_chains = [
    "CC",
    "CCS",
    "CCC(O)=O",
    "CCCC(O)=O",
    "CCC1=CC=CC=C1",
    "C",
    "CCC1=CNC=N1",
    "CC(C)CC",
    "CCCCCN",
    "CSCCC",
    "CCC(N)=O",
    "C1CCCN1",
    "CCCC(N)=O",
    "NC(NCCCC)=N",
    "CCO",
    "CC(O)C",
    "CC(C)C",
    "CCC1=CNC2=C1C=CC=C2",
    "OC1=CC=C(CC)C=C1"
]

side_chain_analogs= [Chem.MolFromSmiles(smi) for smi in methyl_capped_side_chains]

# Filter FreeSolv to find closest neighbors for all AAs

In [ ]:
from rdkit.Chem import AllChem
import tqdm
from rdkit.DataStructs.cDataStructs import BulkTanimotoSimilarity
import os
import numpy as np
import json

def contains_vinyl_group(mol):
    """
    Checks if the molecule contains a double C=CH2 bond (vinyl group).
    """
    # SMARTS pattern for vinyl group: C=C
    vinyl_smarts = "[C]=[CH2]"
    pattern = Chem.MolFromSmarts(vinyl_smarts)

    # Check for substructure match
    return mol.HasSubstructMatch(pattern)

# load molecules from FreeSolv
freesolv_base_path = "/path/to/freesolv/sdf/files" # adjust the path as necessary
free_solv_mols = []

for sdf_file in os.listdir(freesolv_base_path):

    suppl = Chem.SDMolSupplier(os.path.join(freesolv_base_path, sdf_file), removeHs=False)
    mol = next(suppl)
    mol.RemoveAllConformers()
    free_solv_mols.append(mol)
fpgen = AllChem.GetMorganGenerator(radius=3, fpSize=8192)

fingerprints_freesolv= []
fingerprints_aminoacids = []
fingerprints_glycine = []

for mol in free_solv_mols:

    fingerprints_freesolv.append(fpgen.GetFingerprint(mol))

for mol in side_chain_analogs:

    fingerprints_aminoacids.append(fpgen.GetFingerprint(mol))

for mol in [Chem.MolFromSmiles("C(C(=O)O)N")]:

    fingerprints_glycine.append(fpgen.GetFingerprint(mol))
mols_to_calc = set()
mols_to_calc_glycine = set()

for fp in fingerprints_aminoacids:
    
    indices = np.argsort(-np.array(BulkTanimotoSimilarity(fp, fingerprints_freesolv, returnDistance=False)))

    for index in indices:

        if free_solv_mols[index] in mols_to_calc:

            continue
        
        else:
            if not contains_vinyl_group(free_solv_mols[index]):
                mols_to_calc.add(free_solv_mols[index])
                break

for fp in fingerprints_glycine:
    
    indices = np.argsort(-np.array(BulkTanimotoSimilarity(fp, fingerprints_freesolv, returnDistance=False)))

    for i, index in enumerate(indices):

        if i==10:
            break

        mols_to_calc_glycine.add(free_solv_mols[index])

all_mols = mols_to_calc.union(mols_to_calc_glycine)

img = Draw.MolsToGridImage([Chem.RemoveHs(mol) for mol in all_mols], molsPerRow=5, subImgSize=(200,200))

with open("../data/molecules_for_HFE.txt", "w") as file: # adjust the path as necessary

    for mol in all_mols:

        smi = Chem.MolToSmiles(mol)

        file.write(f"{smi}\n")

# Find molecule names in FreeSolv

In [ ]:
def read_molecules(mol_definitions):
    
    molecules = []
    
    for mol_definition in mol_definitions:
        
        molecule = Chem.MolFromSmiles(mol_definition)
            
        molecules.append(molecule)
        
    return molecules

def read_molecules_file(path:str):
    
    with open(path, "r") as file:
        
        all_lines = file.readlines()
        
        molecule_definitions = [line.strip() for line in all_lines if line.strip()!=""]
            
    return molecule_definitions

In [ ]:
from pathlib import Path

EXAMPLES_DIR = Path.cwd().resolve() 
CALIBRATION_DIR = EXAMPLES_DIR.parent
DATA_DIR = CALIBRATION_DIR / "data"

# load molecules from FreeSolv
freesolv_base_path = "/path/to/freesolv/sdf/files" # adjust the path as necessary
free_solv_mols = []
molecules_txt = read_molecules(read_molecules_file(DATA_DIR / "molecules_for_HFE.txt")) # adjust the path as necessary

results = {}

for sdf_file in os.listdir(freesolv_base_path):

    suppl = Chem.SDMolSupplier(os.path.join(freesolv_base_path, sdf_file), removeHs=True)
    mol_free_solv = next(suppl)
    smi_free_solv = Chem.MolToSmiles(mol_free_solv, canonical=True, allHsExplicit=False)
    
    for i, mol_txt in enumerate(molecules_txt):
        
        if Chem.MolToSmiles(mol_txt) == smi_free_solv:
            
            results[f"MOL_{i:03d}"] = sdf_file.replace(".sdf", "")
            

with open(DATA_DIR / "HFE_mols_FreeSolv_conversion.json", "w") as f: # adjust the path as necessary
    
        json.dump(results, f, sort_keys=True, indent=4)